# 📋 [OPS08_M07] 야간 기습 및 다수 드론 동시 출현 극한 상황 모의 역할분담 스트레스 훈련
### AI Co-pilot 협업 기반 소부대 대공 방어 표준운용절차(SOP) 및 모의훈련 워크북 (학생 실습용 워크북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/OPS08/ops08_m07_student.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 드론 경보 시 관측자-지휘자-경계조 역할분담 모의훈련 (Level 2)
- **미션 번호:** OPS08_M07
- **미션 주제:** 야간 기습 및 다수 드론 동시 출현 극한 상황 모의 역할분담 스트레스 훈련
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고 교관 지휘에 엄격히 따른다.


In [ ]:
# [OPS08] 소부대 드론 경보 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np

def verify_mission1_raci(raci_data, checklist_items):
    """미션 1 RACI 매트릭스 및 체크리스트 무결성 검증"""
    print("=" * 60)
    print("📋 [미션 1 검증] 7단계 역할분담 매트릭스 & 체크리스트 검증")
    print("=" * 60)
    
    stages = ["포착", "식별", "보고", "전파", "결심", "대피·대응", "해제"]
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f"❌ 누락된 단계: {s}")
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if "책" in duty or "A" in duty)
            if acc_count != 1:
                print(f"⚠️ [{s} 단계] 책임자(A/책)가 정확히 1명이 아닙니다! (현재 {acc_count}명)")
                stage_pass = False
    
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f"✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 {'[PASS]' if check_pass else '[FAIL]'}")
    print(f"✔ 7단계 RACI 매트릭스 책임자 무결성: {'[PASS]' if stage_pass else '[FAIL]'}")
    
    if stage_pass and check_pass:
        print("\n🎉 [미션 1 PASS] 현장 점검 기준과 역할 분담이 전술 기준에 완벽히 부합합니다!")
    else:
        print("\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 점검하십시오.")

def analyze_mission4_dryrun(log_df):
    """미션 4 1차 모의훈련 타임라인 및 준수율 분석"""
    print("=" * 60)
    print("⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산")
    print("=" * 60)
    
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print("-" * 60)
    print(f"✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {'[PASS]' if avg_lead_time <= 180 else '[FAIL]'}")
    print(f"✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {'[PASS]' if sop_pass_rate >= 80 else '[FAIL]'}")
    
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print("\n🎉 [미션 4 PASS] 1차 Dry-run 목표치를 달성하였습니다! 병목 구간 2건을 AAR에 인계하십시오.")

def grade_mission5_cross_verification(user_answers, answer_key):
    """미션 5 교차확인 판정 채점"""
    print("=" * 60)
    print("🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증")
    print("=" * 60)
    
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get("pred")
        truth = info["truth"]
        is_threat = ("실제 위협" in truth)
        
        is_correct = (user_pred == truth)
        if is_correct:
            correct += 1
        elif is_threat and ("오경보" in str(user_pred)):
            miss_threat += 1
            print(f"🚨 [치명적 오류] {card} 위협을 오경보로 놓쳤습니다! (Miss 발생)")
            
    success_rate = (correct / total) * 100
    print(f"✔ 허위 경보 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)")
    print(f"✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)")
    
    if success_rate >= 85.0 and miss_threat == 0:
        print("\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 완벽히 필터링했습니다!")

def compare_mission9_retrial(m4_data, m9_data):
    """미션 9 1차 대비 2차 훈련 개선율 분석"""
    print("=" * 60)
    print("📈 [미션 9 성과 분석] 1차(M4) 대비 2차 재훈련 반응 속도 & 오류 개선율")
    print("=" * 60)
    
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    
    print(f"1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)")
    print(f"2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)")
    print("-" * 60)
    
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    
    if pass_time and pass_err:
        print("🎉 [미션 9 PASS] 수정 SOP 적용을 통해 팀 단위 전술 반응 속도가 입증되었습니다!")


---

## 🎯 미션 07. 야간 기습 및 다수 드론 동시 출현 극한 상황 모의 역할분담 스트레스 훈련

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> `"야간 조명 제한 및 다수 드론 동시 침투 스트레스 상황에서 지휘권 승계 순서와 10대 위험 방지 조치 이행 타임라인을 구성해줘."`
>
> **🔍 AI 결과 검증 포인트 (HITL):**
> - 비상 절차 전환이 60초 내 완료되었는지, 차선임자 지휘권 승계 선언이 무전망에 명확히 전파되었는지 검토합니다.

---

### 📑 실습자료 및 과업 지침

야간 기습 및 다수 드론 동시 출현 극한 상황 모의 역할분담 스트레스 훈련**

가. 제공 자료 - 스트레스 주입 타임라인 [교육용가상자료] (교관이 T 기준으로 제시)

| **시점** | **주입 내용** | **기대 대응** |
| --- | --- | --- |
| **T+0:00** | 야간, 조명 제한(손전등 사용 금지 가정) | 야간 관측 위치 확인, 요약 카드 확인 |
| **T+1:00** | 3시 방향 윙윙 소리 | 확요 보고, 지휘자 복창 |
| **T+2:00** | 1시·4시·8시에서 드론 3기 동시 출현 | 긴급 모드 전환, 전원 은폐, 표적 우선순위 보고 |
| **T+4:00** | 지휘자 무전 두절(지휘자 역할 퇴장) | 위임 순서에 따라 차선임자 지휘 인수 |
| **T+5:00** | 경계조 1명 넘어짐(모의 경상) | 부상 보고, 응급처치 연계, 인원 확인 |
| **T+7:00** | 인접 분대가 '8시 기체 확인 요청' | 구역 확인 후 관측 결과 회신 |
| **T+9:00** | 2시에서 소리 + 새 떼 소리 혼재 | 교차 확인, 오경보 여부 판정 |
| **T+12:00** | 지휘자 무전 복구 | 차선임자가 지휘자에게 상황 인계(SBAR) |
| **T+14:00** | 해제 명령 | 인원 확인 후 해제 보고 |

나. 제공 자료 - 위험 조치 10개 판정 기준

| **번호** | **위험 조치** | **성공 기준** |
| --- | --- | --- |
| **D1** | 긴급 모드 전환 | 3기 출현 후 1분 이내 '전원 은폐' 전파 |
| **D2** | 은폐 이동 | 전원이 지정 위치로 이동(평가관 확인) |
| **D3** | 표적 우선순위 보고 | 가장 가까운·급강하 표적부터 보고 |
| **D4** | 오사격 방지 확인 | 구두 사격 전 아군·민간인 확인 발언 |
| **D5** | 지휘 위임 | 두절 후 1분 이내 차선임자 '지휘 인수' 선언 |
| **D6** | 부상 보고 | 부상 발생 1분 이내 보고 |
| **D7** | 인원 확인 | 부상 후 및 해제 전 점호 실시 |
| **D8** | 인접 분대 회신 | 요청 2분 이내 관측 결과 회신 |
| **D9** | 오경보 판정 | 교차 확인 후 판정 근거 발언 |
| **D10** | 지휘 복귀 인계 | SBAR 4요소를 포함해 인계 |

다. 작성 서식 1 - 타임라인 기록표(기록자)

| **시점** | **실제 행동·발언** | **행동 시각** | **걸린 시간(초)** | **관련 D번호** |
| --- | --- | --- | --- | --- |
| **T1** |  |  |  |  |
| **T2** |  |  |  |  |
| **T3** |  |  |  |  |
| **T4** |  |  |  |  |
| **T5** |  |  |  |  |
| **T6** |  |  |  |  |
| **T7** |  |  |  |  |
| **T8** |  |  |  |  |
| **T9** |  |  |  |  |

라. 작성 서식 2 - 위험 조치 판정표 및 위임 순서

| **D1** | **D2** | **D3** | **D4** | **D5** | **D6** | **D7** | **D8** | **D9** | **D10** |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |

| **위험 조치 성공률** | 성공 ( ) ÷ 10 × 100 = ( )% → 목표 90% 이상 ( ) |
| --- | --- |

| **비상 절차 전환 시간** | 3기 출현 ( : : ) → 은폐 전파 ( : : ) = ( )초 → 목표 60초 이내 ( ) |
| --- | --- |

| **지휘 위임 순서** | 1순위 지휘자 ( ) → 2순위 ( ) → 3순위 ( ) / 인수 선언 문구: ( ) |
| --- | --- |

마. 안전 통제 점검(훈련 시작 전, 안전 통제관)

| **안전 점검 항목** | **확인** |
| --- | --- |
| 이동 경로에 걸려 넘어질 물체를 치웠다 | ( ) |
| 야간 조명 제한 구역 밖 대기 인원을 지정했다 | ( ) |
| '훈련 중지' 3회 신호를 전원이 알고 있다 | ( ) |
| 모의 부상은 실제 동작 없이 카드로만 표시한다 | ( ) |
| 실탄·공포탄·실제 화기가 없음을 확인했다 | ( ) |

| **점검 항목** | **확인** |
| --- | --- |
| 타임라인 9개 시점을 모두 기록했다 | ( ) |
| 비상 절차 전환 시간을 초 단위로 계산했다 | ( ) |
| 위험 조치 10개를 모두 판정했다 | ( ) |
| 위임이 1분 이내에 이루어졌는지 확인했다 | ( ) |
| 안전 수칙 위반 0건 | ( ) |

**

